# UMUD Attention U-Net and geometry

Separate aponeurosis and fascicle models, followed by geometry for PA, FL and MT.

| Submission | Public score |
|---|---:|
| Colton Downing reference | 0.23551 |
| 20% attention hybrid | **0.23276** |
| Trimmed-length hybrid | 0.23280 |

The hybrid uses 20% attention measurements on 102 images and retains the reference on 207. Physical calibration remains unverified. See [results](https://github.com/rlogger/umud-attention-geometry/blob/main/docs/results.md) and [references](https://github.com/rlogger/umud-attention-geometry/blob/main/docs/references.md).


## Setup

Run from the repository, or upload `umud-attention-geometry.zip` when prompted in Colab. Use a GPU for training or inference. The two saved submission files can be checked on CPU.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

PROJECT = next((p for p in (Path.cwd(), Path.cwd().parent)
                if (p / 'umud_attention').is_dir()), None)
if PROJECT is None:
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload the single source archive: umud-attention-geometry.zip')
    destination = Path('/content/umud_source')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Invalid archive path')
        archive.extractall(destination)
    packages = [p.parent for p in destination.rglob('pyproject.toml')
                if (p.parent/'umud_attention').is_dir()]
    if len(packages) != 1:
        raise ValueError('Expected one source package in the archive')
    PROJECT = packages[0]
if not (PROJECT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Repository files were not found')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT),
                '-r', str(PROJECT/'requirements.txt'), 'kagglehub'], check=True)
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
SOURCE_SHA256 = hashlib.sha256(b''.join(p.read_bytes() for p in sorted((PROJECT/'umud_attention').glob('*.py')))).hexdigest()
print('Project:', PROJECT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
subprocess.run([sys.executable, 'scripts/reproduce_submissions.py', '--output-dir', '/tmp/umud_reproduced'], check=True)


## Data and checkpoints

Restore the existing model ZIP, or select `train` for the fold-0 recipe. Set the ZIP path below; its checksum is included. In Colab, enable `KAGGLE_API_TOKEN` in Secrets for the competition download. The Kaggle account must already have access.

The saved weights are available in the existing [model folder](https://drive.google.com/drive/folders/1dBPdvgnPkDabeDfNdAS_MzLKhNQCNKJV); they are not included in the source repository.


In [ ]:
#@title Data and storage
TRAIN_MODE = "restore" #@param ["restore", "train"]
MODEL_BUNDLE_ZIP = "" #@param {type:"string"}
EXPECTED_SHA256 = "23f87ba200fe5e5ffc1bfc9c8dd86f7b086e02af952b91b698f00d4b4a4df5ff" #@param {type:"string"}
DATA_ROOT = "" #@param {type:"string"}
MOUNT_DRIVE = False #@param {type:"boolean"}
RUN_DIR = "" #@param {type:"string"}
GROUPS_CSV = "" #@param {type:"string"}
COMPETITION = "umud-challenge-muscle-architecture-in-ultrasound-data"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
default_run = Path("/content/drive/MyDrive/UMUD/attention_v1") if MOUNT_DRIVE else PROJECT.parent / "umud_runs" / "attention_v1"
run_dir = Path(RUN_DIR).expanduser() if RUN_DIR.strip() else default_run
run_dir.mkdir(parents=True, exist_ok=True)
restored_bundle = None
if TRAIN_MODE == "restore":
    if not MODEL_BUNDLE_ZIP.strip() or not EXPECTED_SHA256.strip():
        raise ValueError("Restore mode needs MODEL_BUNDLE_ZIP and its separately saved EXPECTED_SHA256.")
    from umud_attention.restore import restore_model_bundle
    restored_bundle = restore_model_bundle(MODEL_BUNDLE_ZIP.strip(), run_dir / "restored_model",
                                           expected_sha256=EXPECTED_SHA256.strip())
    print("Restored saved checkpoints:", restored_bundle["checkpoint_root"])
if DATA_ROOT.strip():
    data_root = Path(DATA_ROOT).expanduser()
else:
    _marker = Path("/content/umud_competition_path.txt")
    _saved_root = _marker.read_text().strip() if _marker.is_file() else ""
    _saved_path = Path(_saved_root) if _saved_root else None
    attached = [Path("/kaggle/input/competitions") / COMPETITION, Path("/kaggle/input") / COMPETITION]
    available = _saved_path if _saved_path is not None and _saved_path.is_dir() else next((p for p in attached if p.is_dir()), None)
    if available is not None:
        data_root = available
    else:
        def _secret(name):
            try:
                from google.colab import userdata
                return userdata.get(name)
            except Exception:
                return None
        _token = _secret("KAGGLE_API_TOKEN")
        _auth_previous = {}
        if _token:
            _auth_previous["KAGGLE_API_TOKEN"] = os.environ.get("KAGGLE_API_TOKEN")
            os.environ["KAGGLE_API_TOKEN"] = _token
        else:
            _username, _key = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
            if not (_username and _key):
                del _username, _key, _token, _auth_previous
                raise RuntimeError("Enable notebook access to KAGGLE_API_TOKEN in Colab Secrets, or set DATA_ROOT to extracted competition data.")
            _auth_previous = {name: os.environ.get(name) for name in ("KAGGLE_USERNAME", "KAGGLE_KEY")}
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = _username, _key
            del _username, _key
        del _token
        try:
            import kagglehub
            data_root = Path(kagglehub.competition_download(COMPETITION))
        except Exception:
            raise RuntimeError("Kaggle download failed. Check secret access, account competition access/rule acceptance, and network availability.") from None
        finally:
            for _name, _previous_value in _auth_previous.items():
                if _previous_value is None:
                    os.environ.pop(_name, None)
                else:
                    os.environ[_name] = _previous_value
            del _auth_previous, _name, _previous_value
if not data_root.is_dir():
    raise FileNotFoundError(data_root)
if Path("/content").is_dir():
    Path("/content/umud_competition_path.txt").write_text(str(data_root))
print("Data:", data_root)
print("Runs:", run_dir)


## Dataset split

Use the saved manifest when restoring. For training, exact image duplicates and horizontal flips share a fold. Subject independence and other near duplicates remain unverified.


In [ ]:
from umud_attention.workflow import prepare_dataset, plot_data_audit
SEED, N_FOLDS = 20261111, 3
if TRAIN_MODE == "restore":
    from umud_attention.data import IMAGE_SUFFIXES, _fingerprint
    from umud_attention.submission import load_expected_ids
    training_manifest_path = Path(restored_bundle["train_manifest"])
    train_manifest = pd.read_csv(training_manifest_path)
    archived_data_paths_available = all(
        Path(path).is_file() for column in ("image_path", "mask_path")
        for path in train_manifest[column])
    # Test-only discovery and identity checks; no training mask audit or new folds.
    _test_paths = sorted(path for path in data_root.rglob("*")
                         if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
                         and ("test_images_v2" in path.relative_to(data_root).parts
                              or data_root.name in {"test_images_v2", "test_set_v2"})
                         and not any(part.startswith(".") or part == "__MACOSX"
                                     for part in path.relative_to(data_root).parts))
    test_manifest = pd.DataFrame({"image_id": [path.name for path in _test_paths],
                                  "image_path": [str(path) for path in _test_paths]})
    _current_ids = load_expected_ids(manifest=test_manifest, expected_count=309)
    _archived_test = pd.read_csv(restored_bundle["test_manifest"])
    _archived_ids = load_expected_ids(manifest=_archived_test, expected_count=309)
    if set(_current_ids) != set(_archived_ids):
        raise ValueError("Current test-image IDs differ from the saved competition cohort.")
    _test_pixels_verified = False
    if "image_hash" in _archived_test:
        if (_archived_test.image_hash.isna().any()
                or _archived_test.image_hash.astype(str).str.strip().eq("").any()):
            raise ValueError("Saved test manifest contains missing image hashes.")
        _expected_hashes = _archived_test.set_index("image_id").image_hash.to_dict()
        _current_hashes = []
        for _row in test_manifest.itertuples():
            _current_hash = _fingerprint(_row.image_path)["image_hash"]
            if _current_hash != _expected_hashes[_row.image_id]:
                raise ValueError(f"Test image pixels changed since the saved run: {_row.image_id}")
            _current_hashes.append(_current_hash)
        test_manifest["image_hash"] = _current_hashes
        _test_pixels_verified = True
    else:
        print("Saved test manifest has no image hashes; only image IDs could be checked.")
    test_manifest.to_csv(run_dir / "test_manifest.csv", index=False)
    (run_dir / "restore_data_summary.json").write_text(json.dumps({
        "mode": "restore", "model_bundle_sha256": restored_bundle["bundle_sha256"],
        "archived_training_manifest": str(training_manifest_path),
        "archived_data_paths_available": archived_data_paths_available,
        "current_test_images": len(test_manifest), "test_pixels_verified": _test_pixels_verified,
        "training_audit_skipped": True}, indent=2))
    print("Training-data audit skipped; current test images:", len(test_manifest))
    print("Test pixels match archived hashes:", _test_pixels_verified)
    print("Archived training paths available for optional evaluation:", archived_data_paths_available)
else:
    train_manifest, test_manifest, fold_counts, audit = prepare_dataset(
        data_root, run_dir, n_folds=N_FOLDS, seed=SEED,
        groups_csv=GROUPS_CSV.strip() or None,
        source_description="Kaggle UMUD competition; actual folder counts recorded below")
    training_manifest_path = run_dir / "train_manifest.csv"
    archived_data_paths_available = True
    display(fold_counts)
    display(pd.Series(audit, name="Data audit"))
    plot_data_audit(train_manifest, per_task=3)
    plt.show()


## Models

ResNet34 encoders with four additive attention gates. Aponeurosis input: 512 pixels; fascicle input: 768 pixels. The recorded run trained fold 0 only.


In [ ]:
#@title Training controls
IMAGE_SIZE = 512 #@param {type:"integer"}
FASC_SIZE = 768 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
GRADIENT_ACCUMULATION = 1 #@param {type:"integer"}
DATA_WORKERS = 4 #@param {type:"integer"}
FASCICLE_CLDICE_WEIGHT = 0.0 #@param {type:"number"}
RUN_PLAIN_CONTROL = False #@param {type:"boolean"}
ALLOW_CPU_TRAINING = False #@param {type:"boolean"}
from umud_attention.workflow import training_summary, plot_history
from dataclasses import asdict
if TRAIN_MODE == "restore":
    checkpoint_root = Path(restored_bundle["checkpoint_root"])
    print("Training skipped; using restored weights:", checkpoint_root)
else:
    from umud_attention.train import TrainConfig, train_fold
    if not torch.cuda.is_available() and not ALLOW_CPU_TRAINING:
        raise RuntimeError("Select a GPU runtime, or explicitly enable CPU training for a small diagnostic run.")
    folds_to_train = [0]
    epochs_this_call = None
    checkpoint_root = run_dir / "attention"
    common = dict(epochs=40, batch_size=BATCH_SIZE,
                  accumulation=GRADIENT_ACCUMULATION, seed=SEED, encoder="resnet34",
                  pretrained=True, workers=DATA_WORKERS, mask_polarity="auto", device="auto")
    configs = [TrainConfig(task=task, fold=fold, attention=True, size=FASC_SIZE if task == "fasc" else IMAGE_SIZE,
                           topology_weight=FASCICLE_CLDICE_WEIGHT if task == "fasc" else 0., **common)
               for fold in folds_to_train for task in ("apo", "fasc")]
    (run_dir / "notebook_run_config.json").write_text(json.dumps({"mode":TRAIN_MODE, "source_sha256":SOURCE_SHA256,
        "max_epochs_this_call":epochs_this_call, "configs":[asdict(c) for c in configs]}, indent=2))
    print("Training", len(configs), "task/fold models; checkpoints:", checkpoint_root)
    for config in configs:
        train_fold(run_dir / "train_manifest.csv", checkpoint_root, config, resume=True, max_epochs_this_call=epochs_this_call)
    if RUN_PLAIN_CONTROL:
        for config in configs:
            plain = TrainConfig(**{**asdict(config), "attention":False})
            train_fold(run_dir / "train_manifest.csv", run_dir / "plain_control", plain, resume=True, max_epochs_this_call=epochs_this_call)


In [ ]:
display(training_summary(checkpoint_root))
plot_history(checkpoint_root)
plt.show()
if RUN_PLAIN_CONTROL and TRAIN_MODE != "restore":
    display(Markdown("**Plain-U-Net control · same folds and training configuration**"))
    display(training_summary(run_dir / "plain_control"))
print("Dice above measures held-out masks; it does not establish a PA/FL/MT score.")


## Geometry check

Optional comparison against geometry derived from paired annotation masks. These are pixel-space references, not independent physical measurements.


In [ ]:
#@title Evaluate held-out mask-derived pixel geometry
RUN_GEOMETRY_EVAL = False #@param {type:"boolean"}
GEOMETRY_EVAL_LIMIT = 0 #@param {type:"integer"}
GEOMETRY_POLICY = "local_image_extent" #@param ["local_image_extent", "legacy_fragment"]
from umud_attention.geometry import GeometryConfig
def geometry_config_for_policy(apo_threshold=0.5, fasc_threshold=0.5):
    settings = dict(apo_threshold=float(apo_threshold), fasc_threshold=float(fasc_threshold))
    if GEOMETRY_POLICY == "local_image_extent":
        settings.update(pennation_mode="local_orientation", fascicle_support_policy="image_extent",
                        max_apo_extrapolation_ratio=0.5, max_outside_image_fraction=0.5)
    elif GEOMETRY_POLICY != "legacy_fragment":
        raise ValueError("Unknown geometry policy")
    return GeometryConfig(**settings)
if RUN_GEOMETRY_EVAL and TRAIN_MODE == "restore" and not archived_data_paths_available:
    print("Geometry evaluation skipped: archived image/mask paths are unavailable. Current test inference can still run.")
elif RUN_GEOMETRY_EVAL:
    from umud_attention.evaluate import evaluate_paired_masks
    from IPython.display import Image as NotebookImage
    paired_report = evaluate_paired_masks(
        training_manifest_path,
        [checkpoint_root / "apo_fold0" / "best.pt", checkpoint_root / "fasc_fold0" / "best.pt"],
        run_dir / "paired_geometry_eval", fold=0, device="auto",
        limit=GEOMETRY_EVAL_LIMIT or None, max_overlays=2,
        config=geometry_config_for_policy())  # evaluator replaces thresholds with checkpoint values
    display(Markdown("**Mask-derived pixel consistency — not physical ground-truth accuracy**"))
    display(pd.Series({k: paired_report[k] for k in (
        "paired_images_available", "paired_images_evaluated", "both_complete_pairs",
        "both_complete_coverage", "excluded_pairs", "unpaired_heldout_images")}))
    display(pd.DataFrame(paired_report["metrics"]))
    for overlay_path in paired_report["overlays"]:
        display(NotebookImage(filename=overlay_path))
    print("Detailed report:", paired_report["report_path"])
else:
    print("Optional geometry check is disabled. Enable RUN_GEOMETRY_EVAL after training.")
print("Geometry policy for evaluation and inference:", GEOMETRY_POLICY)


## Test predictions

The reference is Colton Downing's `Blend 34 ours 66 public CNN 30.csv` (0.23551). Supported measurements receive a fixed 20% contribution. Other rows retain his predictions. The known-ruler adapter uses experimental square-pixel calibration.


In [ ]:
#@title Inference options
CALIBRATION_CSV = "" #@param {type:"string"}
USE_EXPERIMENTAL_KNOWN_RULERS = True #@param {type:"boolean"}
TEAM_REFERENCE_CSV = str(PROJECT / "references/colton_blend34.csv")
NEW_MODEL_BLEND_WEIGHT = 0.2 #@param {type:"number"}
SAVE_PROBABILITY_MASKS = True #@param {type:"boolean"}
from umud_attention.infer import predict_dataset
checkpoint_paths = sorted(checkpoint_root.glob("*/best.pt"))
inference_dir = run_dir / "inference"
inference_report = None
if len(test_manifest) == 0:
    print("No test images found: training-only data; no competition CSV will be created.")
elif len(test_manifest) != 309:
    raise ValueError(f"Expected the full 309-image competition test set, found {len(test_manifest)}.")
else:
    # Read each checkpoint on CPU; use its learned threshold, never a guessed default.
    thresholds_by_task = {"apo": [], "fasc": []}
    for checkpoint_path in checkpoint_paths:
        state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        task = state["config"]["task"]
        if task not in thresholds_by_task or "threshold" not in state:
            raise ValueError(f"Checkpoint lacks a recognized task/learned threshold: {checkpoint_path}")
        threshold = float(state["threshold"])
        if not np.isfinite(threshold) or not 0 < threshold < 1:
            raise ValueError(f"Invalid learned threshold: {checkpoint_path}")
        thresholds_by_task[task].append(threshold)
        del state
    if not all(thresholds_by_task.values()):
        raise ValueError("Inference requires both aponeurosis and fascicle checkpoints.")
    learned_thresholds = {task: float(np.median(values)) for task, values in thresholds_by_task.items()}
    inference_geometry_config = geometry_config_for_policy(learned_thresholds["apo"], learned_thresholds["fasc"])
    print("Geometry policy:", GEOMETRY_POLICY, "· checkpoint threshold medians:", learned_thresholds)
    inference_report = predict_dataset(test_manifest, checkpoint_paths, inference_dir,
        calibration_csv=CALIBRATION_CSV.strip() or None,
        use_known_rulers=USE_EXPERIMENTAL_KNOWN_RULERS,
        reference_csv=TEAM_REFERENCE_CSV.strip() or None,
        blend_weight=NEW_MODEL_BLEND_WEIGHT, save_masks=SAVE_PROBABILITY_MASKS,
        expected_count=309, config=inference_geometry_config)
    display(pd.Series({k:v for k,v in inference_report.items() if k not in ("checkpoints", "geometry_config")}))
    quality = pd.read_csv(inference_dir / "quality.csv")
    display(quality.groupby(["status", "eligible_for_blend"], dropna=False).size().rename("images").reset_index())
    display(quality.loc[quality.status != "ok", ["image_id", "status", "reason"]].head(8))


In [ ]:
if inference_report is not None:
    from umud_attention.data import load_image
    from umud_attention.geometry import plot_geometry
    details = json.loads((inference_dir / "geometry_details.json").read_text())
    examples = [d for d in details if "geometry" in d][:3]
    for item in examples:
        image_path = test_manifest.set_index("image_id").loc[item["image_id"], "image_path"]
        fig, ax = plt.subplots(figsize=(10,5))
        plot_geometry(load_image(image_path), item["geometry"], ax=ax)
        ax.set_title(item["image_id"] + " · " + item["geometry"]["status"])
        plt.show()


## Export

The CSV schema is `image_id,pa_deg,fl_mm,mt_mm`. Review the predictions before submitting on Kaggle.


In [ ]:
#@title View outputs and optionally download submission CSVs
DOWNLOAD_SUBMISSION_CSVS = False #@param {type:"boolean"}
from IPython.display import FileLink
if inference_report:
    for key in ("standalone_submission", "hybrid_submission"):
        if inference_report.get(key):
            display(FileLink(inference_report[key]))
            if DOWNLOAD_SUBMISSION_CSVS:
                try:
                    from google.colab import files
                    files.download(inference_report[key])
                except ImportError:
                    print("Download from the notebook file browser:", inference_report[key])
    for name in ("quality.csv", "inference_report.json"):
        display(FileLink(str(inference_dir / name)))
print("Keep checkpoints, manifests, calibration provenance, and reports together in:", run_dir)
print("No automatic Kaggle submission is performed.")
